In [1]:
import NNSA

# List of available models:

In [2]:
NNSA.available_models()

BaSTIModel (http://basti-iac.oa-abruzzo.inaf.it/)
PARSECModel (https://stev.oapd.inaf.it/PARSEC/)
MISTModel (https://waps.cfa.harvard.edu/MIST/)
SYCLISTModel (https://www.unige.ch/sciences/astro/evolution/en/database/syclist)
DartmouthModel (https://rcweb.dartmouth.edu/stellar/)
YaPSIModel (http://www.astro.yale.edu/yapsi/)


# Load BaSTI model:
#### By default, the age models use sklearn to run the neural networks. If you do not have it installed, or don't want to use it, you can pass the argument 'use_sklearn=False' to switch to a numpy version. Be aware though, in this mode it becomes significantly slower once you run the models on more than ~100 stars.

In [ ]:
age_model = NNSA.BaSTIModel()
#age_model = NNSA.BaSTIModel(use_sklearn=False)

# Estimate the age of a single star using [M/H],MG,BPRP:
#### To estimate the age of a star, use the ages_prediction() method, and pass it a metallicity MH, magnitude MG and color BPRP. Be aware with only these values provided, the return value is a np.array of shape (1,1). The reason will be clear if you check later uses. 

In [10]:
age_model.ages_prediction(MH=0.0,MG=4.0,BPRP=1.0)

array([[13.89808636]])

#### You can use the check_domain() method to verify that the star is within the bounds of the isochrones points used for training the neural networks. The function only checks for this in the HR Diagram, so you pass it a metallicity MH, magnitude MG and color BPRP.

In [8]:
age_model.check_domain(MH=0.0,MG=4.0,BPRP=1.0)

array([ True])

# Estimate the age of a single star using [M/H],MG,BPRP,GBP & GRP:
#### You can also pass the red GRP and blue GBP magnitudes separately as additional inputs. This generally results in better estimations.

In [14]:
age_model.ages_prediction(MH=0.0,MG=2.0,BPRP=1.0,GBP=2.0,GRP=1.0)

array([[1.38048182]])

# Estimate the age of multiple stars:
#### If you pass lists of values of size N as inputs instead of single values, the return value of the ages_prediction() method will be a np.array of shape (N,1). Again, the reason for the 2D shape will be clear by checking later uses.

In [ ]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5])

array([[1.6267248 ],
       [5.99103612]])

In [16]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5],GBP=[2.0,1.0],GRP=[1.0,0.5])

array([[1.38048182],
       [2.16979408]])

#### You can also check if the stars are within the training domain all at once by passing lists of metallicity MH, magnitude MG & color values BPRP.

In [7]:
age_model.check_domain(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5])

array([ True,  True])

#### That way, you can crossmatch both outputs to only keep stars whose age estimates we can reliably trust:

In [16]:
MH = [0.0,-1.0,-2.0,-3.0]
MG = [2.0,3.0,3.0,3.0]
BPRP = [1.0,0.5,0.5,0.5]
GBP = [2.0,1.0,1.0,1.0]
GRP = [1.0,0.5,0.5,0.5]
ages = age_model.ages_prediction(MH=MH,MG=MG,BPRP=BPRP,GBP=GBP,GRP=GRP)
print(ages)
outside_domain = age_model.check_domain(MH=MH,MG=MG,BPRP=BPRP)
print(outside_domain)
ages = ages[outside_domain]
print(ages)

[[1.38048182]
 [2.16979408]
 [2.88745321]
 [3.4155044 ]]
[ True  True  True False]
[[1.38048182]
 [2.16979408]
 [2.88745321]]


# Estimate the age of a single star with n=10 Monte Carlo realisations using uncertainties:
#### If you have uncertainty values for a star, you can pass them to the ages_prediction() method by adding values for eMH, eMG & eBPRP. The idea is that multiple age predictions will be made by offsetting the star's metallicity MH, magnitude MG & color BPRP by adding a random gaussian offset of width (eMH,eMG,eBPRP). The number of realisations made is controlled by the parameter n. The output of the method will then be a np.array of shape (1,n).

In [ ]:
age_model.ages_prediction(MH=0.0,MG=2.0,BPRP=1.0,eMH=0.1,eMG=0.1,eBPRP=0.1,n=10)

array([[2.03544739, 1.59207102, 2.03022214, 1.72963817, 1.74013068,
        2.00467239, 1.67545067, 1.97295733, 2.88586463, 1.85107073]])

#### Once the ages have been predicted for each star n times, you can call statistical methods model.mean_ages(), model.median_ages(), model.mode_ages() and model.std_ages()

In [ ]:
print(age_model.mean_ages())
print(age_model.median_ages())
print(age_model.mode_ages())
print(age_model.std_ages())

[13.70947757]
[13.70947757]
[13.75]
[0.]


# Estimate the age of multiple stars with n=100 Monte Carlo realisations using uncertainties:
#### Finally, you can combine multiple (N) stars age estimations and added uncertainties, with n Monte Carlo realisations. The output of the ages_prediction() method will then be a np.array of shape (N,n).

In [18]:
age_model.ages_prediction(MH=[0.0,-1.0],MG=[2.0,3.0],BPRP=[1.0,0.5],eMH=[0.1,0.1],eMG=[0.1,0.1],eBPRP=[0.1,0.1],n=100)

array([[ 1.41147244,  1.6055406 ,  2.00925307,  2.05047963,  1.85936646,
         5.00034423,  1.87013989,  1.63504486, 18.02710064,  1.82454416,
         5.54986289,  1.73491953,  2.15335699,  1.61798661,  1.63696295,
         2.84202803,  2.27524202,  1.72552032,  6.02683676,  1.55106385,
         3.04771886,  1.43546143,  2.54315543, 13.78988865,  1.97212983,
         1.94652956,  2.41013002,  5.31727163,  1.52240049,  1.77306312,
         3.61537726,  5.37132894,  1.7150251 ,  1.71779586,  1.86685661,
         1.38059539,  1.69890723,  4.15427445,  1.54379498,  1.91644366,
         3.55923463,  2.19561623,  1.39614429, 12.94203455,  1.87397184,
         1.62245476,  1.74526817,  1.83826248,  1.59087206, 11.69699147,
         1.66222136,  2.15502987,  2.01037032,  1.76023033,  2.16793081,
         1.49626312,  9.4520436 ,  1.64149311,  1.68948846,  1.67213471,
         1.95610132, 13.27843678,  1.5020952 , 12.85918827,  2.30699055,
        13.27375121, 14.52022778,  3.84013314,  2.2

In [19]:
print(age_model.mean_ages())
print(age_model.median_ages())
print(age_model.mode_ages())
print(age_model.std_ages())

[3.36217084 5.99660921]
[1.89520775 6.07165351]
[1.65 4.95]
[3.5351525  1.46231221]
